In [1]:
import tkinter as tk
from tkinter import colorchooser
from PIL import Image, ImageTk
import mixbox

CELL_SIZE = 40
STEPS_WB = 5
STEPS_GREY = 10

# initial mixer colours
mixer_white = (255, 255, 255)
mixer_black = (0, 0, 0)
mixer_grey  = (128, 128, 128)

def hex_to_rgb(s):
    s = s.lstrip('#')
    return tuple(int(s[i:i+2], 16) for i in (0,2,4))

def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*[int(c) for c in rgb])

def generate_palette(base_rgb):
    rows = STEPS_WB * 2 + 1
    cols = STEPS_GREY + 1
    img = Image.new("RGB", (cols * CELL_SIZE, rows * CELL_SIZE), "white")
    colors_matrix = []

    global mixer_white, mixer_black, mixer_grey
    white = mixer_white
    black = mixer_black
    grey  = mixer_grey

    for row in range(rows):
        if row < STEPS_WB:
            t = 1 - (row / STEPS_WB)
            mix = mixbox.lerp(base_rgb, white, t)
        elif row > STEPS_WB:
            t = (row - STEPS_WB) / STEPS_WB
            mix = mixbox.lerp(base_rgb, black, t)
        else:
            mix = base_rgb

        row_colors = []
        for col in range(cols):
            if col == 0:
                final = mix
            else:
                t_g = col / STEPS_GREY
                final = mixbox.lerp(mix, grey, t_g)
            row_colors.append(final)
            for y in range(row * CELL_SIZE, (row + 1) * CELL_SIZE):
                for x in range(col * CELL_SIZE, (col + 1) * CELL_SIZE):
                    img.putpixel((x, y), tuple(map(int, final)))

        colors_matrix.append(row_colors)
    return img, colors_matrix

def show_palette(base_rgb):
    img, colors_matrix = generate_palette(base_rgb)
    tk_img = ImageTk.PhotoImage(img)
    panel.config(image=tk_img)
    panel.image = tk_img
    panel.colors_matrix = colors_matrix

def pick_color():
    chosen = colorchooser.askcolor(title="Pick a colour")
    if chosen[1]:
        hexval = chosen[1]
        hex_entry.delete(0, tk.END)
        hex_entry.insert(0, hexval)
        rgb = hex_to_rgb(hexval)
        show_palette(rgb)
        status.config(text=f"Set base to {hexval}")

def pick_mixer(label, target):
    color = colorchooser.askcolor(title=f"Choose {label} mixer")[1]
    if color:
        rgb = hex_to_rgb(color)
        if target == "white":
            globals()['mixer_white'] = rgb
        elif target == "black":
            globals()['mixer_black'] = rgb
        elif target == "grey":
            globals()['mixer_grey'] = rgb
        # re-render palette with same current base
        try:
            rgb_base = hex_to_rgb(hex_entry.get())
            show_palette(rgb_base)
        except:
            pass


def on_hex_enter(event=None):
    s = hex_entry.get().strip()
    if len(s) == 7 and s[0] == '#':
        try:
            rgb = hex_to_rgb(s)
            show_palette(rgb)
            status.config(text=f"Set base to {s}")
        except Exception:
            status.config(text="Invalid hex")
    else:
        status.config(text="Invalid hex")

def on_click(event):
    col = event.x // CELL_SIZE
    row = event.y // CELL_SIZE
    if hasattr(panel, 'colors_matrix'):
        try:
            rgb = panel.colors_matrix[row][col]
            hexval = rgb_to_hex(rgb)

            if event.num == 1:  # Left click: copy
                root.clipboard_clear()
                root.clipboard_append(hexval)
                status.config(text=f"Copied {hexval}")
            elif event.num == 3:  # Right click: set as base
                hex_entry.delete(0, tk.END)
                hex_entry.insert(0, hexval)
                show_palette(rgb)
                status.config(text=f"Set base to {hexval}")

        except IndexError:
            pass


# GUI setup
root = tk.Tk()
root.title("Mixbox Colour Palette")

# inside your layout code
mixer_frame = tk.Frame(root)
mixer_frame.pack(pady=5)

tk.Button(mixer_frame, text="Set white", command=lambda: pick_mixer("white", "white")).pack(side='left', padx=5)
tk.Button(mixer_frame, text="Set black", command=lambda: pick_mixer("black", "black")).pack(side='left', padx=5)
tk.Button(mixer_frame, text="Set grey",  command=lambda: pick_mixer("grey",  "grey" )).pack(side='left', padx=5)

btn = tk.Button(root, text="🎨 Choose Colour", command=pick_color)
btn.pack(pady=5)

hex_entry = tk.Entry(root, font=('Consolas', 12), justify='center', width=10)
hex_entry.pack(pady=2)
hex_entry.bind("<Return>", on_hex_enter)

panel = tk.Label(root)
panel.pack()
panel.bind("<Button-1>", on_click)
panel.bind("<Button-3>", on_click)

status = tk.Label(root, text="Pick or type a hex colour", anchor='w')
status.pack(fill='x')

# Start with default
hex_entry.insert(0, "#e49f23")
show_palette(hex_to_rgb("#e49f23"))

root.mainloop()
